1. 문서의 내용을 읽는다.
2. 문서를 쪼갠다.
  - 토큰수 초과로 답변을 생성하지 못할 수 있고
  - 문서가 길면 (인풋이 길면) 답변 생성이 오래 걸림
3. 임베딩 -> 벡터 데이터베이스에 저장
4. 질문이 있을 때, 벡터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [ ]:
# %pip install --upgrade --quiet docx2txt langchain-community


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader("./tax.docx")
doc = loader.load()

In [5]:
len(doc) # 문서의 길이를 찍어보면 1 이 나온다. 즉 쪼개야 함.

1

일반적으로 문서 쪼갤 때, 랭체인의 Recursive, Character 스플리터를 주로 사용함.
캐릭터는 "\n\n"만 사용 가능한 반면, 리커시브는 "\n\n", "\n", " ", "" 등 사용 가능해서 성능면에서 더 유리.

In [ ]:
# %pip install -qU langchain-text-splitters


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=1500, # 하나의 청크가 관리해주는 글자 수
  chunk_overlap=200 # 청크 간의 겹치는 글자 수
)
# 겹치는 이유? 문서의 연관성을 유지하기 위해

loader = Docx2txtLoader("./tax.docx")
doc_list = loader.load_and_split(text_splitter)

len(doc_list) # 쪼개진 문서의 수

195

In [19]:
from dotenv import load_dotenv
from langchain_ollama import OllamaEmbeddings

load_dotenv()

embeddings = OllamaEmbeddings(model="bge-m3")

In [ ]:
# %pip install langchain-chroma


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [32]:
from langchain_chroma import Chroma

database = Chroma.from_documents(
  documents=doc_list,
  embedding=embeddings,
  collection_name="chroma-tax",
  persist_directory="./chroma"
)

In [23]:
query = "연봉 5천만 원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query, 3) # 유사도 검색을 위한 메서드

retrieved_docs

[Document(id='ecc588c4-fa67-4d47-86f6-2d50308ec644', metadata={'source': './tax.docx'}, page_content='제5관 근로소득공제ㆍ연금소득공제 및 퇴직소득공제 <개정 2009. 12. 31.>\n\n\n\n제47조(근로소득공제) ①근로소득이 있는 거주자에 대해서는 해당 과세기간에 받는 총급여액에서 다음의 금액을 공제한다. 다만, 공제액이 2천만원을 초과하는 경우에는 2천만원을 공제한다. <개정 2012. 1. 1., 2014. 1. 1., 2019. 12. 31.>\n\n\n\n② 일용근로자에 대한 공제액은 제1항에도 불구하고 1일 15만원으로 한다.<개정 2018. 12. 31.>\n\n③ 근로소득이 있는 거주자의 해당 과세기간의 총급여액이 제1항 또는 제2항의 공제액에 미달하는 경우에는 그 총급여액을 공제액으로 한다.\n\n④ 제1항부터 제3항까지의 규정에 따른 공제를 “근로소득공제”라 한다.\n\n⑤ 제1항의 경우에 2인 이상으로부터 근로소득을 받는 사람(일용근로자는 제외한다)에 대하여는 그 근로소득의 합계액을 총급여액으로 하여 제1항에 따라 계산한 근로소득공제액을 총급여액에서 공제한다.<개정 2010. 12. 27.>\n\n⑥ 삭제<2010. 12. 27.>\n\n[전문개정 2009. 12. 31.]\n\n\n\n제47조의2(연금소득공제) ①연금소득이 있는 거주자에 대해서는 해당 과세기간에 받은 총연금액(분리과세연금소득은 제외하며, 이하 이 항에서 같다)에서 다음 표에 규정된 금액을 공제한다. 다만, 공제액이 900만원을 초과하는 경우에는 900만원을 공제한다. <개정 2013. 1. 1.>\n\n\n\n② 제1항에 따른 공제를 “연금소득공제”라 한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제48조(퇴직소득공제) ① 퇴직소득이 있는 거주자에 대해서는 해당 과세기간의 퇴직소득금액에서 제1호의 구분에 따른 금액을 공제하고, 그 금액을 근속연수(1년 미만의 기간이 있는 

In [24]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

In [33]:
# 프롬프트에 페르소나 먼저 적용
prompt = f"""
  - 당신은 최고의 한국 소득세 전문가입니다.
  - [Context]를 참고해서 사용자의 질문에 답변해주세요.

  [Context]
  {retrieved_docs}

  [Question]
  {query}
"""



In [34]:
ai_message = llm.invoke(prompt)

In [35]:
ai_message.text

'제공해주신 법령([Context])을 바탕으로 답변드리겠습니다.\n\n결론부터 말씀드리면, **제공된 자료([Context])에는 구체적인 \'근로소득공제율 표\'와 \'기본세율(과세표준별 세율 구간)\'의 세부 수치가 생략되어 있어, 연봉 5천만 원 직장인의 정확한 소득세 액수를 산출할 수는 없습니다.** \n\n다만, 최고의 소득세 전문가로서 제공된 법령에 근거하여 **소득세가 어떤 과정을 통해 계산되는지 그 구조와 방법**을 상세히 설명해 드리겠습니다.\n\n---\n\n### 연봉 5천만 원 직장인의 소득세 계산 과정 (법령 근거)\n\n직장인(일반 근로자)의 소득세는 총급여액(연봉)에서 시작하여 아래의 단계를 거쳐 최종 세액이 결정됩니다.\n\n#### 1단계: 근로소득공제 차감 (제47조)\n* **총급여액:** 5,000만 원\n* **근로소득공제:** 제47조 제1항에 따라 해당 과세기간에 받는 총급여액에서 법이 정한 공제액을 차감합니다. (공제 한도는 2,000만 원)\n* **결과:** 총급여액(5,000만 원) - 근로소득공제액 = **근로소득금액**\n\n#### 2단계: 종합소득공제 차감 (제6관)\n* 근로소득금액에서 기본공제(본인 및 부양가족 공제) 등 종합소득공제를 추가로 차감합니다.\n* **결과:** 근로소득금액 - 종합소득공제 = **종합소득과세표준** (세금을 매기는 기준 금액)\n\n#### 3단계: 기본세율 적용 (Document 3 / 원천징수세율 규정)\n* Document 3에 따르면 **"근로소득에 대해서는 기본세율을 적용한다"**고 규정되어 있습니다. (일용근로자는 6% 분리과세이나, 질문의 직장인은 일반 근로자로 가정하므로 기본세율 대상입니다.)\n* **결과:** 종합소득과세표준 × 기본세율 = **종합소득산출세액**\n\n#### 4단계: 세액공제 적용 및 연말정산 (Document 2 / 제137조 관련)\n* Document 2(제137조 제1항 제3호)에 따라, 산출된 세액에서 **근로소득세액공제, 

'제공해주신 법령([Context])을 바탕으로 답변드리겠습니다.\n\n결론부터 말씀드리면, **제공된 자료([Context])에는 구체적인 \'근로소득공제율 표\'와 \'기본세율(과세표준별 세율 구간)\'의 세부 수치가 생략되어 있어, 연봉 5천만 원 직장인의 정확한 소득세 액수를 산출할 수는 없습니다.** \n\n다만, 최고의 소득세 전문가로서 제공된 법령에 근거하여 **소득세가 어떤 과정을 통해 계산되는지 그 구조와 방법**을 상세히 설명해 드리겠습니다.\n\n---\n\n### 연봉 5천만 원 직장인의 소득세 계산 과정 (법령 근거)\n\n직장인(일반 근로자)의 소득세는 총급여액(연봉)에서 시작하여 아래의 단계를 거쳐 최종 세액이 결정됩니다.\n\n#### 1단계: 근로소득공제 차감 (제47조)\n* **총급여액:** 5,000만 원\n* **근로소득공제:** 제47조 제1항에 따라 해당 과세기간에 받는 총급여액에서 법이 정한 공제액을 차감합니다. (공제 한도는 2,000만 원)\n* **결과:** 총급여액(5,000만 원) - 근로소득공제액 = **근로소득금액**\n\n#### 2단계: 종합소득공제 차감 (제6관)\n* 근로소득금액에서 기본공제(본인 및 부양가족 공제) 등 종합소득공제를 추가로 차감합니다.\n* **결과:** 근로소득금액 - 종합소득공제 = **종합소득과세표준** (세금을 매기는 기준 금액)\n\n#### 3단계: 기본세율 적용 (Document 3 / 원천징수세율 규정)\n* Document 3에 따르면 **"근로소득에 대해서는 기본세율을 적용한다"**고 규정되어 있습니다. (일용근로자는 6% 분리과세이나, 질문의 직장인은 일반 근로자로 가정하므로 기본세율 대상입니다.)\n* **결과:** 종합소득과세표준 × 기본세율 = **종합소득산출세액**\n\n#### 4단계: 세액공제 적용 및 연말정산 (Document 2 / 제137조 관련)\n* Document 2(제137조 제1항 제3호)에 따라, 산출된 세액에서 **근로소득세액공제, 자녀세액공제, 연금계좌세액공제, 특별세액공제** 등을 차감하여 최종 납부할 소득세(결정세액)를 확정합니다.\n* 기납부한 원천징수세액의 합계가 최종 결정세액을 초과하는 경우에는 그 초과액을 환급받게 됩니다(제137조 제2항).\n\n---\n\n### 요약 및 제언\n정확한 세액을 계산하기 위해서는 **근로소득공제 적용 기준표**와 **종합소득 기본세율 표(6%~45% 구간)**가 추가로 필요합니다. 이 정보가 갖춰진다면 본 법령 구조에 대입하여 원 단위까지 정확한 세액 계산이 가능합니다.'

In [41]:
from langsmith import Client

prompt = Client().pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)


In [42]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

In [ ]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
  llm,
  retriever=database.as_retriever(),
  chain_type_kwargs={"prompt": prompt}
)

In [46]:
ai_message = qa_chain({"query": query})

/var/folders/jk/2pn1bfhj20l75q8kss49wgr00000gn/T/ipykernel_57978/3455095564.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  ai_message = qa_chain({"query": query})


In [47]:
ai_message

{'query': '연봉 5천만 원인 직장인의 소득세는 얼마인가요?',
 'result': '제공된 본문 내용만으로는 연봉 5천만 원인 직장인의 소득세를 산출할 수 없어 정확한 금액을 알 수 없습니다. 소득세 계산에 필요한 소득세율표(기본세율) 및 구체적인 근로소득공제 표 등의 정보가 본문에 포함되어 있지 않습니다.'}